# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata object
metadata = dataset.metadata

print("Dataset title:", metadata.name)
print("Description:\n", metadata.description)
print("Publish date:", metadata.datePublished)
print("Keywords:", metadata.keywords)
print("License:", metadata.license)


## 2. Data Overview
Review available record sets, fields, and their IDs.

### Note
* All references to entities such as record sets, fields, and columns in this notebook will use their `@id` values as required for Croissant datasets.

In [ ]:
# Fetch all record sets from the dataset
record_sets = dataset.record_sets

print("Record Sets Overview:")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']} - Name: {rs.get('name', 'N/A')}")
    # Print available fields and their @id
    fields = rs.get('field', []) if 'field' in rs else []
    if fields:
        print("  Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"    Field @id: {f['@id']} - Name: {f.get('name', 'N/A')}")
            else:
                print(f"    Field @id: {f}")
    print()

# Show example records from each record set
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Example records from RecordSet @id: {rs_id}")
    try:
        count = 0
        for record in dataset.records(record_set=rs_id):
            pprint.pprint(record)
            count += 1
            if count >= 2:  # Show only first 2 records
                break
    except Exception as e:
        print("  Could not load records.", e)
    print("---")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s identified in the previous overview.

Each DataFrame is keyed by its record set `@id`.

In [ ]:
# List all available record set @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"RecordSet @id: {rs_id} - columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"RecordSet @id: {rs_id} is empty.")
    except Exception as e:
        print(f"Could not load records for RecordSet @id: {rs_id}", e)

# Pick first record set for example analysis
example_record_set_id = record_set_ids[0] if record_set_ids else None
example_df = dataframes.get(example_record_set_id, pd.DataFrame())
print("\nExample DataFrame for record set @id:", example_record_set_id)
display(example_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll:
- Select a numeric field by its `@id`.
- Filter records, normalize, and group.

In [ ]:
# Choose a record set and a numeric field for analysis
# We'll try to find a numeric field among available DataFrames

selected_record_set_id = None
numeric_field_id = None
group_field_id = None

for rs_id, df in dataframes.items():
    # Try to detect numeric columns
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        selected_record_set_id = rs_id
        numeric_field_id = numeric_cols[0]
        # Try to get a categorical/group field
        group_cols = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        if group_cols:
            group_field_id = group_cols[0]
        break

if selected_record_set_id and numeric_field_id:
    print(f"Using RecordSet @id: {selected_record_set_id}, Numeric Field: {numeric_field_id}")
    threshold = 0  # Threshold can be adjusted as necessary
    filtered_df = dataframes[selected_record_set_id][dataframes[selected_record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we'll produce histograms and scatter plots for numeric/categorical combinations in the example DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id:
    df = dataframes[selected_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {selected_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Insufficient numeric fields for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrates extraction and analysis of clinical, hormone, and genetic variant data in a highly structured FAIR^2 dataset using the `mlcroissant` library.
- Record sets, fields, and operations reference all entities using their `@id`, as required by Croissant.
- The observed dataset is limited in scope and sample size, and is suitable primarily for academic and illustrative purposes.
- Further statistical or machine learning analysis would require additional data and careful consideration of cohort bias.